# Pipeline ETL: Transformacion de Datos
## World Happiness Report, Global Peace Index, Olympic Results

**Objetivo:** Transformar los datasets originales de Kaggle para normalizarlos al periodo 2015-2019, estandarizar esquemas y producir tres archivos CSV limpios listos para analisis y visualizacion.

**Fuentes originales:**
- [World Happiness Report (2015-2019)](https://www.kaggle.com/datasets/unsdsn/world-happiness)
- [Olympic Summer & Winter Games, 1896-2022](https://www.kaggle.com/datasets/piterfm/olympic-games-medals-19862018)
- [Global Peace Index 2023](https://www.kaggle.com/datasets/ddosad/global-peace-index-2023)


## 1. Importacion de librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 2. Carga de datos originales

Los archivos originales se descargan directamente de Kaggle. El dataset de World Happiness viene dividido en 5 archivos CSV separados (uno por anio), mientras que Global Peace Index y Olympic Results vienen como archivos unicos.

> **Nota:** Subir los archivos originales al entorno de Colab antes de ejecutar esta celda.

In [ ]:
df_2015 = pd.read_csv('2015.csv')
df_2016 = pd.read_csv('2016.csv')
df_2017 = pd.read_csv('2017.csv')
df_2018 = pd.read_csv('2018.csv')
df_2019 = pd.read_csv('2019.csv')

df_peace_raw = pd.read_csv('Global_Peace_Index_2023.csv')
df_olympics_raw = pd.read_csv('olympic_results.csv')

print('Archivos cargados')
print(f'Happiness: {len(df_2015)}+{len(df_2016)}+{len(df_2017)}+{len(df_2018)}+{len(df_2019)} filas')
print(f'Peace: {len(df_peace_raw)} filas')
print(f'Olympics: {len(df_olympics_raw)} filas')

## 3. Analisis Exploratorio de Datos (EDA)

Antes de transformar, inspeccionamos la estructura de cada dataset para identificar inconsistencias en nombres de columnas, tipos de datos, valores nulos y rangos temporales.

### 3.1 World Happiness Report: esquemas por anio

In [ ]:
happiness_files = {
    2015: df_2015, 2016: df_2016, 2017: df_2017, 2018: df_2018, 2019: df_2019
}

for year, df in happiness_files.items():
    print(f'{year} ({len(df)} filas, {len(df.columns)} columnas)')
    print(f'  Columnas: {list(df.columns)}')
    print(f'  Nulls: {df.isnull().sum().sum()}')
    print()

### 3.2 Global Peace Index: estructura y cobertura temporal

In [ ]:
print(f'Shape: {df_peace_raw.shape}')
print(f'Columnas: {list(df_peace_raw.columns)}')
print(f'Anios disponibles: {sorted(df_peace_raw["year"].unique())}')
print(f'Paises unicos: {df_peace_raw["Country"].nunique()}')
print(f'\nNulls por columna:')
print(df_peace_raw.isnull().sum())
df_peace_raw.head()

### 3.3 Olympic Results: estructura y cobertura temporal

In [ ]:
print(f'Shape: {df_olympics_raw.shape}')
print(f'Columnas: {list(df_olympics_raw.columns)}')
print(f'Juegos unicos: {sorted(df_olympics_raw["slug_game"].unique())}')
print(f'Paises unicos: {df_olympics_raw["country_name"].nunique()}')
print(f'\nNulls por columna:')
print(df_olympics_raw.isnull().sum())
df_olympics_raw.head()

## 4. Transformacion: World Happiness Report

### Problemas identificados:
1. Los 5 archivos tienen **esquemas diferentes** (nombres de columnas distintos entre anios)
2. Los anios 2017-2019 **no incluyen la columna Region**
3. Los anios 2018-2019 **no incluyen Dystopia Residual**
4. No existe una columna `Year` explicita en ningun archivo

### Estrategia:
- Normalizar todas las columnas a un esquema unificado de 11 campos
- Agregar la columna `Year` a cada archivo
- Concatenar los 5 DataFrames

In [ ]:
def normalize_2015(df):
    df = df.copy()
    df['Year'] = 2015
    df = df.rename(columns={'Economy (GDP per Capita)':'GDP per Capita','Family':'Social Support',
        'Health (Life Expectancy)':'Health Life Expectancy','Trust (Government Corruption)':'Trust Government Corruption'})
    return df[['Country','Year','Happiness Rank','Happiness Score','GDP per Capita','Social Support',
               'Health Life Expectancy','Freedom','Generosity','Trust Government Corruption','Dystopia Residual']]

def normalize_2016(df):
    df = df.copy()
    df['Year'] = 2016
    df = df.rename(columns={'Economy (GDP per Capita)':'GDP per Capita','Family':'Social Support',
        'Health (Life Expectancy)':'Health Life Expectancy','Trust (Government Corruption)':'Trust Government Corruption'})
    return df[['Country','Year','Happiness Rank','Happiness Score','GDP per Capita','Social Support',
               'Health Life Expectancy','Freedom','Generosity','Trust Government Corruption','Dystopia Residual']]

def normalize_2017(df):
    df = df.copy()
    df['Year'] = 2017
    df = df.rename(columns={'Happiness.Rank':'Happiness Rank','Happiness.Score':'Happiness Score',
        'Economy..GDP.per.Capita.':'GDP per Capita','Family':'Social Support',
        'Health..Life.Expectancy.':'Health Life Expectancy','Trust..Government.Corruption.':'Trust Government Corruption',
        'Dystopia.Residual':'Dystopia Residual'})
    return df[['Country','Year','Happiness Rank','Happiness Score','GDP per Capita','Social Support',
               'Health Life Expectancy','Freedom','Generosity','Trust Government Corruption','Dystopia Residual']]

def normalize_2018(df):
    df = df.copy()
    df['Year'] = 2018
    df = df.rename(columns={'Overall rank':'Happiness Rank','Score':'Happiness Score','GDP per capita':'GDP per Capita',
        'Social support':'Social Support','Healthy life expectancy':'Health Life Expectancy',
        'Freedom to make life choices':'Freedom','Perceptions of corruption':'Trust Government Corruption',
        'Country or region':'Country'})
    df['Dystopia Residual'] = np.nan
    return df[['Country','Year','Happiness Rank','Happiness Score','GDP per Capita','Social Support',
               'Health Life Expectancy','Freedom','Generosity','Trust Government Corruption','Dystopia Residual']]

def normalize_2019(df):
    df = df.copy()
    df['Year'] = 2019
    df = df.rename(columns={'Overall rank':'Happiness Rank','Score':'Happiness Score','GDP per capita':'GDP per Capita',
        'Social support':'Social Support','Healthy life expectancy':'Health Life Expectancy',
        'Freedom to make life choices':'Freedom','Perceptions of corruption':'Trust Government Corruption',
        'Country or region':'Country'})
    df['Dystopia Residual'] = np.nan
    return df[['Country','Year','Happiness Rank','Happiness Score','GDP per Capita','Social Support',
               'Health Life Expectancy','Freedom','Generosity','Trust Government Corruption','Dystopia Residual']]

df_happiness = pd.concat([normalize_2015(df_2015), normalize_2016(df_2016), normalize_2017(df_2017),
                          normalize_2018(df_2018), normalize_2019(df_2019)], ignore_index=True)

print(f'Dataset consolidado: {df_happiness.shape}')
print(f'Filas por anio:\n{df_happiness["Year"].value_counts().sort_index()}')

### 4.1 Verificacion de tipos de datos y valores nulos

In [ ]:
print('Tipos de datos:')
print(df_happiness.dtypes)
print(f'\nValores nulos por columna:')
print(df_happiness.isnull().sum())

numeric_cols = ['Happiness Score','GDP per Capita','Social Support',
                'Health Life Expectancy','Freedom','Generosity',
                'Trust Government Corruption','Dystopia Residual']
for col in numeric_cols:
    df_happiness[col] = pd.to_numeric(df_happiness[col], errors='coerce')
df_happiness['Year'] = df_happiness['Year'].astype(int)
df_happiness['Happiness Rank'] = df_happiness['Happiness Rank'].astype(int)

print('\nTipos despues de conversion:')
print(df_happiness.dtypes)

### 4.2 Estadisticas descriptivas

In [ ]:
df_happiness.describe()

### 4.3 Verificacion de integridad

In [ ]:
duplicados = df_happiness.groupby(['Country','Year']).size()
duplicados_multi = duplicados[duplicados > 1]
print(f'Duplicados (Country, Year): {len(duplicados_multi)}')
if len(duplicados_multi) > 0:
    print(duplicados_multi)
print(f'\nRango de anios: {df_happiness["Year"].min()} - {df_happiness["Year"].max()}')
print(f'Paises unicos: {df_happiness["Country"].nunique()}')
print(f'Total filas: {len(df_happiness)}')
df_happiness.head(10)

### 4.4 EDA visual: distribuciones del World Happiness Report

Se explora visualmente la distribucion de las variables numericas del dataset consolidado.

In [ ]:
num_cols = ['Happiness Score','GDP per Capita','Social Support',
            'Health Life Expectancy','Freedom','Generosity','Trust Government Corruption']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    df_happiness[col].dropna().hist(bins=30, ax=axes[i], color='#5b9bd5', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Frecuencia', fontsize=8)
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Distribucion de variables numericas (World Happiness)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.4.1 Boxplots por variable para detectar outliers

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(data=df_happiness, y=col, ax=axes[i], color='#5b9bd5', width=0.4, fliersize=3)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].set_ylabel('')
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Boxplots por variable (World Happiness)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.4.2 Evolucion del Happiness Score promedio por anio

In [ ]:
yearly_avg = df_happiness.groupby('Year')['Happiness Score'].mean()
fig, ax = plt.subplots(figsize=(8, 4))
yearly_avg.plot(kind='bar', ax=ax, color='#5b9bd5', edgecolor='white', width=0.6)
ax.set_title('Happiness Score promedio por anio', fontsize=12, fontweight='bold')
ax.set_xlabel('Anio')
ax.set_ylabel('Happiness Score promedio')
ax.set_xticklabels(yearly_avg.index.astype(int), rotation=0)
for i, v in enumerate(yearly_avg):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 4.4.3 Distribucion del Happiness Score por anio (violin plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=df_happiness, x='Year', y='Happiness Score', ax=ax, palette='muted', inner='quartile', cut=0)
ax.set_title('Distribucion del Happiness Score por anio', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.4.4 Top 10 y Bottom 10 paises por Happiness Score (promedio 2015-2019)

In [ ]:
avg_by_country = df_happiness.groupby('Country')['Happiness Score'].mean().sort_values(ascending=False)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
avg_by_country.head(10).plot(kind='barh', ax=ax1, color='#34d399', edgecolor='white')
ax1.set_title('Top 10 paises mas felices', fontsize=11, fontweight='bold')
ax1.set_xlabel('Happiness Score promedio')
ax1.invert_yaxis()
avg_by_country.tail(10).plot(kind='barh', ax=ax2, color='#ef4444', edgecolor='white')
ax2.set_title('Top 10 paises menos felices', fontsize=11, fontweight='bold')
ax2.set_xlabel('Happiness Score promedio')
ax2.invert_yaxis()
plt.tight_layout()
plt.show()

### 4.4.5 Relacion entre GDP per Capita y Happiness Score

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(df_happiness['GDP per Capita'], df_happiness['Happiness Score'],
                     c=df_happiness['Year'], cmap='viridis', alpha=0.5, s=20, edgecolors='white', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Anio')
ax.set_title('GDP per Capita vs Happiness Score', fontsize=12, fontweight='bold')
ax.set_xlabel('GDP per Capita')
ax.set_ylabel('Happiness Score')
plt.tight_layout()
plt.show()

## 5. Transformacion: Global Peace Index

### Problemas identificados:
1. El dataset original abarca 2008-2023, necesitamos solo 2015-2019
2. Contiene la columna `iso3c` que no necesitamos

### Estrategia:
- Filtrar filas al rango temporal 2015-2019
- Eliminar columna `iso3c`

In [ ]:
df_peace = df_peace_raw[(df_peace_raw['year'] >= 2015) & (df_peace_raw['year'] <= 2019)].copy()
if 'iso3c' in df_peace.columns:
    df_peace.drop(columns=['iso3c'], inplace=True)
df_peace.reset_index(drop=True, inplace=True)

print(f'Shape original: {df_peace_raw.shape} -> Shape filtrado: {df_peace.shape}')
print(f'Anios: {sorted(df_peace["year"].unique())}')
print(f'Paises unicos: {df_peace["Country"].nunique()}')

### 5.1 Verificacion de valores nulos y tipos de datos

In [ ]:
print('Tipos de datos:')
print(df_peace.dtypes)
print(f'\nValores nulos:')
print(df_peace.isnull().sum())
nulls = df_peace[df_peace.isnull().any(axis=1)]
if len(nulls) > 0:
    print(f'\nFilas con valores nulos:')
    print(nulls.to_string())

### 5.2 Estadisticas descriptivas

In [ ]:
df_peace.describe()

### 5.3 EDA visual: distribuciones del Global Peace Index

In [ ]:
peace_cols = ['Overall Scores','Safety and Security','Ongoing Conflict','Militarian']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, col in enumerate(peace_cols):
    df_peace[col].dropna().hist(bins=30, ax=axes[i], color='#818cf8', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Frecuencia', fontsize=8)
fig.suptitle('Distribucion de variables numericas (Global Peace Index)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.3.1 Overall Peace Score promedio por anio

In [ ]:
peace_yearly = df_peace.groupby('year')['Overall Scores'].mean()
fig, ax = plt.subplots(figsize=(8, 4))
peace_yearly.plot(kind='bar', ax=ax, color='#818cf8', edgecolor='white', width=0.6)
ax.set_title('Overall Peace Score promedio por anio', fontsize=12, fontweight='bold')
ax.set_xlabel('Anio')
ax.set_ylabel('Overall Scores promedio')
ax.set_xticklabels(peace_yearly.index.astype(int), rotation=0)
for i, v in enumerate(peace_yearly):
    ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 5.3.2 Top 10 paises mas pacificos y menos pacificos

In [ ]:
peace_avg = df_peace.groupby('Country')['Overall Scores'].mean().sort_values()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
peace_avg.head(10).plot(kind='barh', ax=ax1, color='#34d399', edgecolor='white')
ax1.set_title('Top 10 paises mas pacificos', fontsize=11, fontweight='bold')
ax1.set_xlabel('Overall Scores (menor = mas pacifico)')
ax1.invert_yaxis()
peace_avg.tail(10).plot(kind='barh', ax=ax2, color='#ef4444', edgecolor='white')
ax2.set_title('Top 10 paises menos pacificos', fontsize=11, fontweight='bold')
ax2.set_xlabel('Overall Scores')
ax2.invert_yaxis()
plt.tight_layout()
plt.show()

### 5.3.3 Boxplot comparativo de componentes del indice de paz

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_peace_melted = df_peace.melt(id_vars=['Country','year'], value_vars=peace_cols, var_name='Componente', value_name='Score')
sns.boxplot(data=df_peace_melted, x='Componente', y='Score', ax=ax, palette='Set2')
ax.set_title('Distribucion de cada componente del indice de paz', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 6. Transformacion: Olympic Results

### Problemas identificados:
1. El dataset contiene todos los Juegos Olimpicos desde 1896 hasta 2022
2. El anio esta embebido en `slug_game` (ej: `rio-2016`, `pyeongchang-2018`)
3. Los unicos Juegos dentro del rango 2015-2019 son Rio 2016 y PyeongChang 2018

### Estrategia:
- Extraer el anio de `slug_game` mediante regex
- Filtrar al rango 2015-2019

In [ ]:
df_olympics_raw['year_extracted'] = df_olympics_raw['slug_game'].str.extract(r'(\d{4})').astype(int)
print(f'Anios disponibles: {sorted(df_olympics_raw["year_extracted"].unique())}')
print(f'Dentro de 2015-2019: {((df_olympics_raw.year_extracted>=2015)&(df_olympics_raw.year_extracted<=2019)).sum()}')
print(f'Fuera de rango: {((df_olympics_raw.year_extracted<2015)|(df_olympics_raw.year_extracted>2019)).sum()}')

In [ ]:
df_olympics = df_olympics_raw[
    (df_olympics_raw['year_extracted'] >= 2015) &
    (df_olympics_raw['year_extracted'] <= 2019)
].copy()
df_olympics.drop(columns=['year_extracted'], inplace=True)
df_olympics.reset_index(drop=True, inplace=True)

print(f'Shape original: {df_olympics_raw.shape} -> Shape filtrado: {df_olympics.shape}')
print(f'Juegos incluidos: {df_olympics["slug_game"].unique()}')
print(f'Paises unicos: {df_olympics["country_name"].nunique()}')

### 6.1 Verificacion de valores nulos y tipos de datos

In [ ]:
print('Tipos de datos:')
print(df_olympics.dtypes)
print(f'\nValores nulos por columna:')
print(df_olympics.isnull().sum())

### 6.2 Distribucion de medallas

In [ ]:
print('Distribucion de medal_type:')
print(df_olympics['medal_type'].value_counts(dropna=False))
print(f'\nDistribucion por juego:')
print(df_olympics['slug_game'].value_counts())

### 6.3 EDA visual: distribuciones de Olympic Results

In [ ]:
medal_counts = df_olympics[df_olympics['medal_type'].notna()]['medal_type'].value_counts()
colors_medal = {'GOLD':'#fbbf24','SILVER':'#94a3b8','BRONZE':'#b45309'}
fig, ax = plt.subplots(figsize=(6, 4))
medal_counts.plot(kind='bar', ax=ax, color=[colors_medal.get(m,'#5b9bd5') for m in medal_counts.index],
                  edgecolor='white', width=0.5)
ax.set_title('Cantidad de registros por tipo de medalla', fontsize=12, fontweight='bold')
ax.set_ylabel('Cantidad')
ax.set_xticklabels(medal_counts.index, rotation=0)
for i, v in enumerate(medal_counts):
    ax.text(i, v + 5, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 6.3.1 Top 15 paises por total de medallas

In [ ]:
medals = df_olympics[df_olympics['medal_type'].notna()]
medals_by_country = medals.groupby('country_name')['medal_type'].value_counts().unstack(fill_value=0)
medals_by_country['Total'] = medals_by_country.sum(axis=1)
top15 = medals_by_country.nlargest(15, 'Total')

medal_types = ['GOLD','SILVER','BRONZE']
colors_stack = ['#fbbf24','#94a3b8','#b45309']
existing = [m for m in medal_types if m in top15.columns]
existing_c = [colors_stack[medal_types.index(m)] for m in existing]

fig, ax = plt.subplots(figsize=(12, 6))
top15[existing].plot(kind='barh', stacked=True, ax=ax, color=existing_c, edgecolor='white')
ax.set_title('Top 15 paises por medallas (Rio 2016 + PyeongChang 2018)', fontsize=12, fontweight='bold')
ax.set_xlabel('Total de medallas')
ax.set_ylabel('')
ax.invert_yaxis()
ax.legend(title='Tipo', loc='lower right')
plt.tight_layout()
plt.show()

### 6.3.2 Disciplinas y participacion por juego

In [ ]:
disc_count = df_olympics.groupby('slug_game')['discipline_title'].nunique()
part_count = df_olympics.groupby('slug_game').size()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
disc_count.plot(kind='bar', ax=ax1, color=['#fb923c','#38bdf8'], edgecolor='white', width=0.4)
ax1.set_title('Disciplinas por juego', fontsize=11, fontweight='bold')
ax1.set_ylabel('Disciplinas')
ax1.set_xticklabels(disc_count.index, rotation=0)
for i, v in enumerate(disc_count):
    ax1.text(i, v+0.3, str(v), ha='center', fontsize=10)

part_count.plot(kind='bar', ax=ax2, color=['#fb923c','#38bdf8'], edgecolor='white', width=0.4)
ax2.set_title('Total de registros por juego', fontsize=11, fontweight='bold')
ax2.set_ylabel('Registros')
ax2.set_xticklabels(part_count.index, rotation=0)
for i, v in enumerate(part_count):
    ax2.text(i, v+30, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 7. Resumen comparativo de los tres datasets transformados

Verificacion final de que los tres datasets estan alineados al periodo 2015-2019.

In [ ]:
print('RESUMEN DE DATASETS TRANSFORMADOS')
print()
print(f'World Happiness Report')
print(f'  Filas: {len(df_happiness)} | Columnas: {len(df_happiness.columns)}')
print(f'  Anios: {sorted(df_happiness["Year"].unique())}')
print(f'  Paises: {df_happiness["Country"].nunique()}')
print(f'\nGlobal Peace Index')
print(f'  Filas: {len(df_peace)} | Columnas: {len(df_peace.columns)}')
print(f'  Anios: {sorted(df_peace["year"].unique())}')
print(f'  Paises: {df_peace["Country"].nunique()}')
print(f'\nOlympic Results')
print(f'  Filas: {len(df_olympics)} | Columnas: {len(df_olympics.columns)}')
print(f'  Juegos: {list(df_olympics["slug_game"].unique())}')
print(f'  Paises: {df_olympics["country_name"].nunique()}')

## 8. Analisis de Correlacion por Dataset

Para cada tabla se calcula la **matriz de correlacion de Pearson** entre las variables numericas. Esto permite identificar pares de variables altamente correlacionadas (|r| > 0.8), que son candidatas a eliminacion para reducir la redundancia.

> **Criterio de decision:** Si dos variables tienen |r| > 0.8, se considera eliminar una de ellas.

In [ ]:
def correlation_analysis(df, title, threshold=0.8):
    numeric_df = df.select_dtypes(include=[np.number])
    if numeric_df.shape[1] < 2:
        print(f'{title}: menos de 2 variables numericas, no se puede calcular correlacion.')
        return None
    corr = numeric_df.corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
    ax.set_title(f'Matriz de Correlacion: {title}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    pairs = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            r = corr.iloc[i,j]
            if abs(r) >= threshold:
                pairs.append({'Variable 1':corr.columns[i],'Variable 2':corr.columns[j],'r':round(r,4)})
    if pairs:
        print(f'Pares con |r| >= {threshold}:')
        print(pd.DataFrame(pairs).sort_values('r', key=abs, ascending=False).to_string(index=False))
    else:
        print(f'No se encontraron pares con |r| >= {threshold}')
    return corr

### 8.1 Correlacion: World Happiness Report

In [ ]:
corr_happiness = correlation_analysis(df_happiness, 'World Happiness Report')

### 8.2 Correlacion: Global Peace Index

In [ ]:
corr_peace = correlation_analysis(df_peace, 'Global Peace Index')

### 8.3 Correlacion: Olympic Results

In [ ]:
corr_olympics = correlation_analysis(df_olympics, 'Olympic Results')

### 8.4 Resumen de variables candidatas a eliminacion

Con base en los resultados anteriores, se identifican las variables redundantes y se justifica cual eliminar de cada par correlacionado.

In [ ]:
print('RESUMEN DE ANALISIS DE CORRELACION')
print()
threshold = 0.8
datasets = {'World Happiness':df_happiness, 'Global Peace Index':df_peace, 'Olympic Results':df_olympics}
for name, df in datasets.items():
    numeric_df = df.select_dtypes(include=[np.number])
    if numeric_df.shape[1] < 2:
        print(f'{name}: insuficientes variables numericas')
        continue
    corr = numeric_df.corr()
    count = 0
    print(f'{name} ({numeric_df.shape[1]} variables numericas)')
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            r = corr.iloc[i,j]
            if abs(r) >= threshold:
                print(f'  {corr.columns[i]} <-> {corr.columns[j]}: r = {r:.4f}')
                count += 1
    if count == 0:
        print(f'  Sin pares altamente correlacionados')
    print()

## 9. Eliminacion de variables redundantes

Con base en el analisis de correlacion, se eliminan las siguientes variables:

**World Happiness Report:**
- `Happiness Rank`: redundante con `Happiness Score` (r = -0.99). El ranking es simplemente el score ordenado; se conserva Score por ser la variable continua de mayor utilidad analitica.

**Global Peace Index:**
- `Safety and Security`: redundante con `Overall Scores` (r = 0.92)
- `Ongoing Conflict`: redundante con `Overall Scores` (r = 0.91)

Se conservan `Overall Scores` (metrica principal de paz) y `Militarian` (r = 0.59, por debajo del umbral, aporta una dimension independiente).

**Olympic Results:**
- No aplica: el dataset no cuenta con suficientes variables numericas correlacionables.

In [ ]:
df_happiness.drop(columns=['Happiness Rank'], inplace=True)
print(f'World Happiness: eliminada "Happiness Rank", {df_happiness.shape[1]} columnas restantes')
print(f'  Columnas: {list(df_happiness.columns)}')

df_peace.drop(columns=['Safety and Security','Ongoing Conflict'], inplace=True)
print(f'\nGlobal Peace Index: eliminadas "Safety and Security" y "Ongoing Conflict", {df_peace.shape[1]} columnas restantes')
print(f'  Columnas: {list(df_peace.columns)}')

print('\nVariables redundantes eliminadas correctamente')

## 10. Exportacion de datasets transformados

Se exportan los tres datasets como archivos CSV independientes, listos para ser cargados en Power BI u otras herramientas de visualizacion.

In [ ]:
df_happiness.to_csv('world_happiness_2015_2019.csv', index=False, sep=';')
df_peace.to_csv('Global_Peace_Index_2015_2019.csv', index=False, sep=';')
df_olympics.to_csv('olympic_results_2015_2019.csv', index=False)

print('Archivos exportados:')
print('  world_happiness_2015_2019.csv')
print('  Global_Peace_Index_2015_2019.csv')
print('  olympic_results_2015_2019.csv')